In [1]:
import sys
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "results/assemblyworldbench/benchmark/benchmark.json").exists()
)
RESULTS = ROOT / "results"
PAPER = ROOT.parent / "AssemblyWorldBench"
CACHE = ROOT / "notebooks/.cache/paper-analysis"
RECOMPUTE = False
WORKERS = 6
sys.path.insert(0, str(ROOT / "notebooks"))
import paper_analysis as analysis  # noqa: E402

analysis.configure(RESULTS, PAPER, CACHE)

# Geometric refinement: evidence audit

All experiment inputs come from the exported `results/` package. No historical run directories or temporary scratchpad are read. The geometry analysis uses the existing keyed evaluator-input interface to resolve dataset-side point clouds and target poses. Derived statistics and caches remain in `notebooks/.cache/paper-analysis/`. PDF figures are written to the sibling paper's `fig/` directories. Run with the project environment (`uv sync --extra episodes --group inspection`).

Intervals are pointwise shape-paired, source-stratified bootstrap intervals; they are not simultaneous significance claims. Missing input records are reported explicitly, never replaced with numbers from a report.

In [2]:
coverage = analysis.audit_available()
display(coverage["missing"]["garf_hybrid"])
display(coverage["candidate_files"])

{'status': 'missing',
 'required': 'Standard and agent-initialized GARF per-object outputs with checkpoint, seeds and shared protocol'}

[]

## Available diagnostic versus missing refinement runs
The existing Fantastic Breaks GARF-style evaluation scores agent-only poses; it is not a standard-GARF or hybrid run. A valid paired experiment requires the same object manifest, point clouds, anchor, checkpoint, refinement schedule, seeds, per-object poses and scores, initialization checks, and separate agent/GPU costs. Until those records exist under results, the hybrid table remains explicitly pending.

In [3]:
directory = RESULTS / coverage["garf_diagnostic"]
print("Existing agent-only diagnostic files:")
print("\n".join(str(p.relative_to(RESULTS)) for p in sorted(directory.rglob("*")) if p.is_file()))

Existing agent-only diagnostic files:
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/initial_validation_errors.json
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/mcp_call_statistics.json
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/meta.json
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/metrics.jsonl
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/metrics_summary.json
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/report.md
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/validation.json
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/verification-retry-01--01048.json
fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf/verification-retry-09--09003.json


## Verify the available agent-only diagnostic
These checks verify object/archive identity, recorded sampling counts, anchor membership, and aggregate arithmetic. A surface-sampling seed is not a refinement seed. The package does not contain matched refinement checkpoints or flow-seed outputs.

In [4]:
import numpy as np

records = analysis.read_jsonl(directory / "metrics.jsonl")
summary = analysis.read_json(directory / "metrics_summary.json")
block = directory.parent.parent
run = analysis.read_json(block / "run.json")
assert {r["sample_id"] for r in records} == set(run["samples"])
assert len(records) == len({r["sample_id"] for r in records}) == 150
anchors = []
for row in records:
    episode = block / "samples" / row["sample_id"].replace("/", "--") / "final.episode.zip"
    assert analysis.digest(episode) == row["episode_sha256"]
    assert sum(p["points"] for p in row["parts"]) == 5000
    assert all(p["points"] >= 20 for p in row["parts"])
    anchor = next(p for p in row["parts"] if p["part_id"] == row["anchor_part_id"])
    assert anchor["points"] == max(p["points"] for p in row["parts"])
    assert abs(anchor["RMSE_R"]) < 1e-5 and abs(anchor["RMSE_T"]) < 1e-7
    assert row["source_scale_divisor"] >= 1 and row["task_scale_divisor"] > 0
    anchors.append({"sample_id": row["sample_id"], "anchor": row["anchor_part_id"]})
means = {m: float(np.mean([r[m] for r in records])) for m in ["PA", "CD", "RMSE_R", "RMSE_T"]}
assert all(np.isclose(means[m], summary[m], atol=1e-9) for m in means)
audit = {
    "objects": len(records),
    "archive_hashes_verified": len(records),
    "points_per_object": 5000,
    "surface_sampling_seeds": sorted({r["sampling_seed"] for r in records}),
    "means": means,
    "anchors": anchors,
    "status": "Agent-only diagnostic verified at record level; matched refinement unavailable",
    "missing_refinement_evidence": ["standard and hybrid outputs", "checkpoint identity", "shared point-cloud identity", "flow seeds", "initialization self-checks", "refinement cost"],
}
analysis.save_json(CACHE / "refinement_audit.json", audit)
display({k: v for k, v in audit.items() if k != "anchors"})

{'objects': 150,
 'archive_hashes_verified': 150,
 'points_per_object': 5000,
 'surface_sampling_seeds': [42],
 'means': {'PA': 0.9166666666666666,
  'CD': 0.006980433387845825,
  'RMSE_R': 11.45318904054547,
  'RMSE_T': 0.025427781580396817},
 'status': 'Agent-only diagnostic verified at record level; matched refinement unavailable',
 'missing_refinement_evidence': ['standard and hybrid outputs',
  'checkpoint identity',
  'shared point-cloud identity',
  'flow seeds',
  'initialization self-checks',
  'refinement cost']}